This is a Synthetic ticket generator + latency + token tracking


## Business Context

Companies often need realistic support data to:

- Test AI workflows
- Evaluate LLM quality before deployment
- Measure latency and cost
- Compare business outcomes vs technical metrics

Instead of using real customer data, we generate synthetic enterprise support tickets using GPT-4.

## What This Notebook Does

1. Generates synthetic support tickets.
2. Summarizes them.
3. Measures:
   - Latency
   - Token usage
   - Estimated cost
4. Returns structured evaluation results.

This mimics a lightweight internal AI evaluation pipeline.

In [ ]:
!pip install openai python-dotenv --quiet

In [ ]:
import time
import os
import json
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI

In [ ]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in .env file")

client = OpenAI(api_key=api_key)
MODEL = "gpt-4"

generate tickets

In [ ]:
def generate_tickets(n: int):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": "You generate realistic enterprise SaaS support tickets."
            },
            {
                "role": "user",
                "content": f"Generate {n} detailed enterprise support tickets."
            }
        ]
    )

    return response

evaluate metrics

In [ ]:
def summarize_tickets(tickets_text: str):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": f"Summarize these support tickets:\n\n{tickets_text}"
            }
        ]
    )
    return response

In [ ]:
def run_evaluation(n=3):
    start_time = time.time()

    # Step 1: Generate
    tickets_response = generate_tickets(n)
    tickets_text = tickets_response.choices[0].message.content

    # Step 2: Summarize
    summary_response = summarize_tickets(tickets_text)
    summary_text = summary_response.choices[0].message.content

    # Metrics
    total_tokens = (
        tickets_response.usage.total_tokens +
        summary_response.usage.total_tokens
    )

    latency = round(time.time() - start_time, 2)
    cost_estimate = estimate_cost(total_tokens)

    return {S
        "tickets": tickets_text,
        "summary": summary_text,
        "latency_seconds": latency,
        "tokens_used": total_tokens,
        "estimated_cost_usd": cost_estimate
    }

test the code 

In [ ]:
results = run_evaluation(3)
results